# Text Clustering Guide

This notebook demonstrates how to cluster a set of strings by their metrics, specifically handling:
- Case-insensitivity.
- Ignoring specific stop words without modifying the original dataset.
- Defining a specific number of classes ($k$).
- Mapping the clustered labels back to the original text.

## Approach 1: TF-IDF with K-Means

The most efficient way to handle case-insensitivity and "ignore words" without losing your original strings is to use `TfidfVectorizer`. It natively converts text to lowercase, filters out a provided list of stop words, and transforms the text into numerical metrics (TF-IDF features). 

Because this happens internally in the vectorizer, the original DataFrame remains untouched, and we simply append the predicted cluster labels as a new column.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

def cluster_strings(df, text_column, k, ignore_words=None):
    """
    Clusters a pandas Series of strings into k classes.
    """
    # TfidfVectorizer handles case-insensitivity (lowercase=True by default) 
    # and ignores specified words via the stop_words parameter.
    vectorizer = TfidfVectorizer(stop_words=ignore_words, lowercase=True)
    
    # Convert strings to numerical metric vectors
    X = vectorizer.fit_transform(df[text_column])
    
    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    
    # Map the resulting cluster IDs directly back to the original DataFrame
    df['Cluster'] = kmeans.fit_predict(X)
    
    return df

# ==========================================
# USE CASE: 20 Strings mapped into 3 classes
# ==========================================
data = [
    # Topic A: Space
    "The moon is orbiting Earth beautifully",
    "NASA launches a SUPER rocket to Mars",
    "Space exploration and galaxy stars",
    "Rocket engine testing in deep space",
    "Astronauts landing on the moon",
    "Solar system and distant planets",
    "Telescope observing the SUPER galaxy",
    
    # Topic B: Football
    "The striker scored a beautiful goal",
    "Football match ends in a draw",
    "Championship tournament finals",
    "Players training on the football pitch",
    "Goalkeeper makes a stunning save",
    "Referee blows the final whistle",
    "Team captain lifting the SUPER trophy",
    
    # Topic C: Baking/Cooking
    "Baking a delicious chocolate cake",
    "Chef preparing a gourmet meal",
    "Recipe for the perfect roasted chicken",
    "Mixing ingredients for the dough",
    "Oven temperature for baking bread",
    "Chopping fresh vegetables for SUPER soup"
]

df = pd.DataFrame(data, columns=["Raw_Text"])

# Words to ignore (Notice we include 'super' which appears across completely different topics)
words_to_ignore = ['the', 'a', 'in', 'on', 'and', 'for', 'to', 'is', 'super']

# Cluster the data (k=3)
clustered_df = cluster_strings(df, text_column="Raw_Text", k=3, ignore_words=words_to_ignore)

# Display the results grouped by cluster
for cluster_id in sorted(clustered_df['Cluster'].unique()):
    print(f"--- Cluster {cluster_id} ---")
    cluster_texts = clustered_df[clustered_df['Cluster'] == cluster_id]['Raw_Text']
    for text in cluster_texts:
        print(f"  - {text}")
    print()

## Approach 2: Extracting Class Names Automatically

To generate class names without an LLM, the standard approach is to extract the **Top-N keywords** from the K-Means cluster centers (centroids). Since `TfidfVectorizer` represents text as a matrix of word weights, each cluster centroid inherently contains the highest-weighted words for that specific group.

In [ ]:
def cluster_strings_with_names(df, text_column, k, ignore_words=None, top_n_words=3):
    """
    Clusters strings and automatically generates a label based on centroid weights.
    """
    vectorizer = TfidfVectorizer(stop_words=ignore_words, lowercase=True)
    X = vectorizer.fit_transform(df[text_column])
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    df['Cluster_Code'] = kmeans.fit_predict(X)
    
    # ---------------------------------------------------------
    # GENERATE CLASS NAMES FROM CENTROIDS
    # ---------------------------------------------------------
    feature_names = vectorizer.get_feature_names_out()
    centroids = kmeans.cluster_centers_
    
    cluster_mapping = {}
    
    for cluster_id in range(k):
        # Sort the centroid weights to find the highest-scoring words for this cluster
        top_indices = centroids[cluster_id].argsort()[-top_n_words:][::-1]
        top_words = [feature_names[idx].title() for idx in top_indices]
        
        # Create a combined name (e.g., "Moon / Galaxy / Space")
        class_name = " / ".join(top_words)
        cluster_mapping[cluster_id] = class_name
        
    # Map the generated names back to the DataFrame
    df['Cluster_Name'] = df['Cluster_Code'].map(cluster_mapping)
    
    return df, cluster_mapping

# Execute with the same data
clustered_df_named, labels = cluster_strings_with_names(df, "Raw_Text", 3, words_to_ignore)

print("--- GENERATED CLASS MAPPING ---")
for code, name in labels.items():
    print(f"Code {code} -> {name}")

print("\n--- CLUSTERED DATA ---")
print(clustered_df_named[['Cluster_Code', 'Cluster_Name', 'Raw_Text']])

## Approach 3: Semantic Clustering (Sentence Embeddings)

Instead of relying on word frequencies (TF-IDF), you can use **Sentence Transformers**. It uses pre-trained neural networks to convert the entire string into a dense mathematical vector that represents *semantic meaning*, not just exact words. 

**Why use it over TF-IDF:**
* TF-IDF treats "Astronaut" and "Cosmonaut" as completely unrelated words because they are spelled differently.
* Sentence Embeddings know that "Astronaut" and "Cosmonaut" mean the same thing and will plot them next to each other in mathematical space.
* It handles typos, synonyms, and completely different phrasing effortlessly.

In [ ]:
# Note: To run this block, you need to install sentence-transformers:
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd

# 1. Load a fast, pre-trained local model
# model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Convert strings to semantic embeddings
# embeddings = model.encode(df['Raw_Text'].tolist())

# 3. Cluster the embeddings
# kmeans = KMeans(n_clusters=3, random_state=42)
# df['Semantic_Cluster'] = kmeans.fit_predict(embeddings)

# print(df[['Semantic_Cluster', 'Raw_Text']])